# 01 — Découpage des séances en prises de parole (*chunks*)

Objectif : extraire des séances OCR + NER du JORF des *chunks* (une prise de parole + sa rubrique) en vue d'un plongement vectoriel.

Règles :
- une prise de parole commence à une entité `SPK` valide (« M. X. », « M. le président. »…) et court jusqu'à la `SPK` valide suivante ;
- un intertitre (bloc `Section-Header` ou bloc portant une entité `TITL`) clôt la prise de parole en cours : le texte qui suit sans nouvel orateur est ignoré (ordre du jour, etc.) ;
- la rubrique est la dernière entité `TITL` rencontrée ; les `Section-Header` sans `TITL` qui la suivent sont les sous-rubriques ;
- en-têtes/pieds de page et images ne coupent rien et ne sont pas repris dans le texte.

Sortie : `output/<chambre>/<ark>/chunk_NNNN.json`.

In [ ]:
import json
import re
from dataclasses import dataclass, field
from datetime import datetime, timezone

import pandas as pd

from debats import OUTPUT_DIR, Block, Entity, Session, html_to_text, iter_sessions, load_session, session_paths

## Une séance

In [ ]:
session = load_session(session_paths()[1])
session

In [ ]:
pd.Series([b.label for b in session.blocks]).value_counts()

## Détection des orateurs et des rubriques

Une `SPK` n'ouvre une prise de parole que si elle est précédée d'une civilité (« M. », « Mme », « M. le »…) en début de paragraphe, ou après une fin de phrase (paragraphes fusionnés par l'OCR : « M. Leblanc. Non ! M. Borgeot. … »). Sinon c'est une simple mention (« … demande à M. le ministre »).

In [ ]:
SKIPPED_LABELS = {"Page-Header", "Page-Footer", "Image", "Caption", "Footnote"}
SECTION_LABEL = "Section-Header"
# Une TITL dans un bloc Text n'est un intertitre que si elle couvre l'essentiel du bloc.
TITLE_MIN_COVERAGE = 0.5

# Civilité (et article) précédant le nom de l'orateur, balises <b> comprises.
HONORIFIC_RE = re.compile(
    r"(?:<b>\s*)?(?:MM?\.|Mmes?|Mlle)(?:\s|</?b>)*(?:(?:le|la)\s+|l['’]\s*)?(?:</?b>\s*)*$"
)
SENTENCE_END_RE = re.compile(r"[.!?)…]$")
PARAGRAPH_START_RE = re.compile(r"<(?:p|br|li|td)\b[^>]*>$", re.IGNORECASE)


def speaker_cut(block: Block, spk: Entity) -> int | None:
    """Offset HTML où commence la prise de parole introduite par `spk`, ou None si simple mention."""
    prefix = block.html[: spk.start]
    m = HONORIFIC_RE.search(prefix)
    if m is None:
        return None
    head = prefix[: m.start()].rstrip()
    if not head or PARAGRAPH_START_RE.search(head):
        return m.start()
    before = html_to_text(head).rsplit("\n", 1)[-1]
    if not SENTENCE_END_RE.search(before):
        return None
    return m.start()


def section_title(block: Block) -> str | None:
    titles = block.entities_of("TITL")
    if not titles:
        return None
    title = " ".join(" ".join(t.text.split()) for t in titles)
    if block.label != SECTION_LABEL and len(title) < TITLE_MIN_COVERAGE * len(block.text):
        return None
    return title

In [ ]:
spks = [(b, spk) for b in session.blocks for spk in b.entities_of("SPK")]
pd.DataFrame(
    {"spk": spk.text, "valide": speaker_cut(b, spk) is not None, "contexte": b.text[:120]}
    for b, spk in spks[:20]
)

## Découpage

In [ ]:
@dataclass
class Turn:
    spk: Entity
    section: str | None
    subsections: list[str]
    blocks: list[str] = field(default_factory=list)
    parts: list[str] = field(default_factory=list)

    def add(self, block: Block, fragment: str) -> None:
        text = html_to_text(fragment)
        if not text:
            return
        if block.uuid not in self.blocks:
            self.blocks.append(block.uuid)
        if self.parts and (self.parts[-1][-1].islower() or self.parts[-1][-1] in ",;"):
            # Paragraphe coupé entre deux blocs (changement de colonne ou de page).
            self.parts[-1] = f"{self.parts[-1]} {text}"
        else:
            self.parts.append(text)

    @property
    def text(self) -> str:
        return "\n".join(self.parts)


def split_turns(session: Session) -> tuple[list[Turn], dict]:
    turns: list[Turn] = []
    stats = {"rejected_spk": 0, "orphan_blocks": 0}
    section: str | None = None
    subsections: list[str] = []
    current: Turn | None = None

    def close() -> None:
        nonlocal current
        if current is not None and current.parts:
            turns.append(current)
        current = None

    for block in session.blocks:
        if block.label in SKIPPED_LABELS:
            continue

        title = section_title(block)
        if title is not None:
            close()
            section, subsections = title, []
            continue
        if block.label == SECTION_LABEL:
            close()
            if subtitle := block.text.replace("\n", " "):
                subsections.append(subtitle)
            continue

        cuts: list[tuple[int, Entity]] = []
        for spk in block.entities_of("SPK"):
            cut = speaker_cut(block, spk)
            if cut is None:
                stats["rejected_spk"] += 1
            elif not cuts or cut > cuts[-1][0]:
                cuts.append((cut, spk))

        head_end = cuts[0][0] if cuts else len(block.html)
        if current is not None:
            current.add(block, block.html[:head_end])
        elif html_to_text(block.html[:head_end]):
            stats["orphan_blocks"] += 1

        for i, (cut, spk) in enumerate(cuts):
            close()
            end = cuts[i + 1][0] if i + 1 < len(cuts) else len(block.html)
            current = Turn(spk=spk, section=section, subsections=list(subsections))
            current.add(block, block.html[cut:end])

    close()
    return turns, stats

In [ ]:
turns, stats = split_turns(session)
print(len(turns), "prises de parole", stats)
pd.DataFrame(
    {"orateur": t.spk.text, "rubrique": t.section, "sous-rubriques": t.subsections, "texte": t.text[:150]}
    for t in turns[:15]
)

## Toutes les séances

In [ ]:
created = datetime.now(timezone.utc).isoformat(timespec="seconds")


def chunk_record(session: Session, turn: Turn) -> dict:
    return {
        "session": session.name,
        "chamber": session.chamber_name,
        "date": session.date.isoformat(),
        "created": created,
        "blocks": turn.blocks,
        "spk_uuid": turn.spk.uuid,
        "content": {"section": turn.section, "subsections": turn.subsections, "text": turn.text},
    }


rows, chunks = [], []
for session in iter_sessions():
    turns, stats = split_turns(session)
    chunks.append((session, [chunk_record(session, t) for t in turns]))
    rows.append({"séance": session.name, "chambre": session.chamber, "chunks": len(turns), **stats})

summary = pd.DataFrame(rows)
print(summary["chunks"].sum(), "chunks pour", len(summary), "séances")
summary

## Écriture

In [ ]:
for session, records in chunks:
    session_dir = OUTPUT_DIR / session.chamber / session.ark
    session_dir.mkdir(parents=True, exist_ok=True)
    for stale in session_dir.glob("chunk_*.json"):
        stale.unlink()
    for i, record in enumerate(records, start=1):
        (session_dir / f"chunk_{i:04d}.json").write_text(
            json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
        )
print("→", OUTPUT_DIR)